# Клиенты торгового эквайринга за месяц

Одна строка — один договор SA. Периметр секции 01 `final_df`: договор пересекает месяц, к нему привязан действующий `cf_ter_type = 'P'`.

Колонки: ИНН, ЦФТ ID, номер договора, дата открытия, дата закрытия, РФ. Пустая дата закрытия — договор на этот месяц ещё открыт. РФ приводится к виду `Чувашский РФ`, как `normalize_filial_rf` в `final_df`. ИНН из 9 цифр дополняется слева до 10, из 11 — до 12.

Смените `REPORT_MONTH` и запустите обе ячейки. Файл `trade_acquiring_clients_YYYY-MM.xlsx` ляжет в `/home/jovyan/documents/Equaring`.



In [ ]:
import getpass
import re
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

REPORT_MONTH = '2026-05'
OUT_DIR = Path('/home/jovyan/documents/Equaring')

report_ts = pd.to_datetime(REPORT_MONTH + '-01')
month_start = report_ts.strftime('%Y-%m-%d')
month_end = (report_ts + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
print(REPORT_MONTH, month_start, month_end)

LAKE_USER = input('Логин Impala: ').strip()
LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')

user_params = {'user_name': LAKE_USER, 'password': LAKE_PASSWORD}

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params=user_params,
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)


In [ ]:
def normalize_filial_rf(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\s+', ' ', str(v)).strip()
    m = re.search(r'(?i)рф', s)
    if m:
        s = s[:m.end()]
    s = re.sub(r'(?i)\b(ао|пао|оао|зао)\b.*$', '', s).strip(' ,;.-')
    if not s:
        return None
    s = s[:1].upper() + s[1:].lower()
    return re.sub(r'(?i)рф', 'РФ', s)


sql = '''-- Договоры SA за месяц: номер, даты, ИНН. 9 цифр -> 10, 11 -> 12.
WITH sa AS (
  SELECT DISTINCT
    a.n_agr,
    a.c_agr_number,
    a.d_valid_from,
    a.d_valid_to,
    CASE length(regexp_replace(c.c_inn, '[^0-9]', ''))
      WHEN 9 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 10, '0')
      WHEN 11 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 12, '0')
      ELSE regexp_replace(c.c_inn, '[^0-9]', '')
    END AS inn
  FROM ods_alpha.scd1_agreements a
  JOIN ods_alpha.scd1_companies c
    ON c.n_cmp = a.n_cmp_client
  WHERE upper(trim(a.acq_class)) = 'SA'
    AND a.d_valid_from <= '{month_end}'
    AND (a.d_valid_to IS NULL OR a.d_valid_to >= '{month_start}')
    AND coalesce(a.ods_deleted_flg, '0') <> '1'
    AND coalesce(c.ods_deleted_flg, '0') <> '1'
    AND c.c_inn IS NOT NULL
    AND EXISTS (
      SELECT 1
      FROM ods_alpha.scd1_agr_terms t
      WHERE t.n_agr = a.n_agr
        AND t.d_valid_from <= '{month_end}'
        AND (t.d_valid_to IS NULL OR t.d_valid_to > '{month_start}')
        AND upper(trim(t.cf_ter_type)) = 'P'
        AND coalesce(t.ods_deleted_flg, '0') <> '1'
    )
),
-- Одна живая карточка OCRM на ИНН, самая новая.
ocrm AS (
  SELECT inn, row_id
  FROM (
    SELECT
      s.inn,
      soe.row_id,
      row_number() OVER (
        PARTITION BY s.inn
        ORDER BY soe.created DESC, soe.row_id DESC
      ) AS rn
    FROM (SELECT DISTINCT inn FROM sa) s
    JOIN ocrm_ul.s_org_ext soe
      ON CASE length(regexp_replace(soe.x_inn, '[^0-9]', ''))
           WHEN 9 THEN lpad(regexp_replace(soe.x_inn, '[^0-9]', ''), 10, '0')
           WHEN 11 THEN lpad(regexp_replace(soe.x_inn, '[^0-9]', ''), 12, '0')
           ELSE regexp_replace(soe.x_inn, '[^0-9]', '')
         END = s.inn
    WHERE soe.x_removed_flg = 'N'
      AND soe.x_duplicate_flg = 'N'
  ) t
  WHERE rn = 1
),
-- Идентификатор CDI по карточке OCRM.
cdi AS (
  SELECT inn, cdi_id
  FROM (
    SELECT
      o.inn,
      e.party_id AS cdi_id,
      row_number() OVER (PARTITION BY o.inn ORDER BY e.party_id DESC) AS rn
    FROM ocrm o
    JOIN cdiul.ext_id_org e
      ON e.cmo_ext_party_source_id = o.row_id
     AND e.cmo_ext_source_system LIKE 'OCRM%'
  ) t
  WHERE rn = 1
),
-- ЦФТ ID той же организации в CDI.
cft AS (
  SELECT c.inn, e.cmo_ext_party_source_id AS cft_id
  FROM cdi c
  JOIN cdiul.ext_id_org e
    ON e.party_id = c.cdi_id
   AND e.cmo_ext_source_system LIKE 'CFT%'
),
-- Подпись филиала по ЦФТ: мерчант -> подразделение -> филиал.
filial AS (
  SELECT cft_id, filial_rf
  FROM (
    SELECT
      c.cft_id,
      br.c_shortlabel AS filial_rf,
      row_number() OVER (
        PARTITION BY c.cft_id
        ORDER BY CASE WHEN m.c_depart IS NULL THEN 1 ELSE 0 END, m.id DESC
      ) AS rn
    FROM (SELECT DISTINCT cft_id FROM cft) c
    JOIN ods.scd1_z_r2_ip_merchants m
      ON cast(m.c_cl_org AS string) = c.cft_id
    LEFT JOIN ods.scd1_z_depart dep ON dep.id = m.c_depart
    LEFT JOIN ods.scd1_z_branch br ON br.id = dep.c_filial
    WHERE coalesce(m.ods_deleted_flg, '0') <> '1'
  ) t
  WHERE rn = 1
),
-- Одна строка на договор. Если ЦФТ несколько, берётся тот, где филиал заполнен.
picked AS (
  SELECT
    s.inn,
    c.cft_id,
    s.c_agr_number,
    s.d_valid_from,
    s.d_valid_to,
    f.filial_rf,
    row_number() OVER (
      PARTITION BY s.n_agr
      ORDER BY CASE WHEN f.filial_rf IS NULL THEN 1 ELSE 0 END, c.cft_id
    ) AS rn
  FROM sa s
  LEFT JOIN cft c ON c.inn = s.inn
  LEFT JOIN filial f ON f.cft_id = c.cft_id
)
-- Шесть полей выгрузки.
SELECT
  inn,
  cft_id,
  c_agr_number,
  d_valid_from,
  d_valid_to,
  filial_rf
FROM picked
WHERE rn = 1'''
sql = sql.format(month_start=month_start, month_end=month_end)

with imp:
    imp.execute('set MEM_LIMIT=16g')
    df = imp.fetch(sql)

df = df.copy()
df.columns = [
    'ИНН',
    'ЦФТ ID',
    'Номер договора',
    'Дата открытия договора',
    'Дата закрытия договора',
    'РФ, в котором обслуживается клиент',
]
df['РФ, в котором обслуживается клиент'] = df['РФ, в котором обслуживается клиент'].apply(normalize_filial_rf)
df['ИНН'] = df['ИНН'].map(lambda v: None if pd.isna(v) else str(v).strip())

out = OUT_DIR / ('trade_acquiring_clients_%s.xlsx' % REPORT_MONTH)
out.parent.mkdir(parents=True, exist_ok=True)
df.to_excel(out, index=False, sheet_name='договоры', engine='openpyxl')
print('строк', len(df))
print(out)
df.head(20)


## Импорт Excel

Читает записанный файл. ИНН и ЦФТ ID загружаются как текст, чтобы ведущий ноль не снялся при открытии.


In [ ]:
excel_df = pd.read_excel(
    out,
    sheet_name='договоры',
    dtype={'ИНН': str, 'ЦФТ ID': str},
)
print('файл', out)
print('строк', len(excel_df))
excel_df.head(20)
